# 📌 Traccia: Regressione - SVR vs Gradient Boosting con Tecniche di Feature Selection
- Tipo di problema: Regressione
- Dataset: diabetes da sklearn.datasets (~440 campioni, 10 feature numeriche)

1. Pipeline 1:
    - Preprocessing: StandardScaler
    - Riduzione dimensionalità: SelectKBest con f_regression
    - Modello: SVR (kernel RBF, ottimizzazione di C e gamma)

2. Pipeline 2:
    - Preprocessing: RobustScaler (più robusto a outlier)
    - Riduzione dimensionalità: PCA
    - Modello: GradientBoostingRegressor (ottimizzazione di n_estimators e max_depth)

- Metrica di valutazione: R² (score di determinazione)

- Valutazione tramite Nested Cross-Validation:
    - Outer CV: 5-fold
    - Inner CV: 3-fold per tuning iperparametri delle pipeline

✅ Obiettivo dello studente:
Costruire le due pipeline descritte, applicare nested cross-validation, confrontare le performance in termini di R², e giustificare quale pipeline si comporta meglio sul dataset dato.

## Import del dataset

In [16]:
from sklearn.datasets import load_diabetes

X, y = load_diabetes(return_X_y = True)

X.shape

(442, 10)

## Pipeline 1
- Preprocessing: StandardScaler
- Riduzione dimensionalità: SelectKBest con f_regression
- Modello: SVR (kernel RBF, ottimizzazione di C e gamma)

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.svm import SVR

pipeline_svr = Pipeline([
    ("scaler", StandardScaler()),
    ("featureSelection", SelectKBest()),
    ("svr", SVR())
])

params_grid_svr = {
    "featureSelection__score_func": [f_regression, mutual_info_regression],
    "svr__kernel": ["rbf", "poly"],
    "svr__C": [1, 0.75, 1.25],
    "svr__gamma": ["scale", "auto"]
}

## Pipeline 2
- Preprocessing: RobustScaler (più robusto a outlier)
- Riduzione dimensionalità: PCA
- Modello: GradientBoostingRegressor (ottimizzazione di n_estimators e max_depth)

In [18]:
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingRegressor

pipeline_gbr = Pipeline([
    ("scaler", RobustScaler()),
    ("pca", PCA()),
    ("gbr", GradientBoostingRegressor())
])

params_grid_gbr = {
    "pca__n_components": [2, 3, 5],
    "pca__svd_solver": ["auto", "full", "randomized"] ,
    "gbr__n_estimators": [75, 100, 125],
    "gbr__max_depth": [3, 10, None]
}

## Nested Cross-Validation

In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
)

def nested_cv(model, param_grid, X_train, y_train,
              outer_splits=5, inner_splits=5,
              scoring: list[str] = None,
              random_state=42, verbose=True):

    # Assicurati che `y` sia un array 1D
    if isinstance(y_train, pd.DataFrame):  # Se è un DataFrame Pandas
        y_train = y_train.values.ravel()
    elif isinstance(y_train, pd.Series):  # Se è una Serie Pandas
        y_train = y_train.values
    else:  # Se è un array Numpy
        y_train = np.ravel(y_train)

    # Determina il tipo di task di apprendimento automatico
    est_type = getattr(model, "_estimator_type", None)
    is_clf = (est_type == "classifier")

    if scoring is None:
        if is_clf:
            scoring = ['accuracy', 'roc_auc']
        else:
            scoring = ['r2', 'mae', 'rmse']

    # CROSS-VALIDATION ESTERNA
    outer_cv = KFold(n_splits=outer_splits, shuffle=True, random_state=random_state)

    # Dizionari per salvare i risultati
    score_results = {metric: [] for metric in scoring}
    all_fold_best_params = []

    for outer_fold, (train_fold_idx, val_idx) in enumerate(outer_cv.split(X_train), 1):
        if verbose:
            print(f"\nPerforming Outer Fold {outer_fold}/{outer_splits}")

        # Usare il metodo .iloc per X, se è un DataFrame
        if isinstance(X_train, pd.DataFrame):
            X_train_fold, X_val = X_train.iloc[train_fold_idx], X_train.iloc[val_idx]
        else:  # Altrimenti usa indicizzazione standard
            X_train_fold, X_val = X_train[train_fold_idx], X_train[val_idx]

        y_train_fold, y_val = y_train[train_fold_idx], y_train[val_idx]

        # --- 3. CICLO DI CROSS-VALIDATION INTERNA (TUNING) ---
        inner_cv = KFold(n_splits=inner_splits, shuffle=True, random_state=random_state)
        primary_metric = scoring[0]  # GridSearchCV ottimizza per la prima metrica della lista

        if verbose:
            print(f"Performing GridSearchCV (optimizing for '{primary_metric}')...")

        grid_search = GridSearchCV(model, param_grid, cv=inner_cv, n_jobs=-1, scoring=primary_metric)
        grid_search.fit(X_train_fold, y_train_fold)

        # Salva i migliori parametri per questo fold
        all_fold_best_params.append(grid_search.best_params_)
        if verbose:
            print(f"  Best Params for this fold: {grid_search.best_params_}")

        # --- 4. VALUTAZIONE SUL TEST SET ESTERNO ---
        best_model_for_fold = grid_search.best_estimator_
        y_pred = best_model_for_fold.predict(X_val)

        if verbose:
            print("  Calculating metrics on the outer test set...")

        # Calcola e salva tutte le metriche richieste
        for metric in scoring:
            if metric == 'accuracy':
                score = accuracy_score(y_val, y_pred)
            elif metric == 'roc_auc':
                try:
                    y_score = best_model_for_fold.predict_proba(X_val)[:, 1]
                    score = roc_auc_score(y_val, y_score)
                except (AttributeError, IndexError):
                    score = np.nan # Modello non ha predict_proba o è binario/monoclasse
            elif metric == 'r2':
                score = r2_score(y_val, y_pred)
            elif metric == 'mae':
                score = mean_absolute_error(y_val, y_pred)
            elif metric == 'mse':
                score = mean_squared_error(y_val, y_pred)
            elif metric == 'rmse':
                score = root_mean_squared_error(y_val, y_pred)
            else:
                score = np.nan # Metrica non riconosciuta

            score_results[metric].append(score)
            if verbose:
                print(f"    {metric.upper()}: {score:.4f}")

    # --- 5. RIEPILOGO FINALE ---
    if verbose:
        print("\n--- Nested Cross-Validation Final Report ---")

    final_summary = {}
    for metric, scores in score_results.items():
        mean_score = np.nanmean(scores)
        std_score = np.nanstd(scores)
        final_summary[metric] = {
            'mean': mean_score,
            'std': std_score,
            'all_scores': scores
        }
        if verbose:
            print(f"Final {metric.upper()} estimate: {mean_score:.4f} ± {std_score:.4f}")

    return {
        'performance_summary': final_summary,
        'all_fold_best_params': all_fold_best_params
    }

# Train, test split

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

## Nested CV su Pipeline 1

In [21]:
import time

start_time_svr = time.time()

result_svr = nested_cv(
    model = pipeline_svr,
    param_grid = params_grid_svr,
    X_train = X_train,
    y_train = y_train,
    outer_splits = 5,
    inner_splits = 3,
    scoring = ["r2"]
)

end_time_svr = time.time()


Performing Outer Fold 1/5
Performing GridSearchCV (optimizing for 'r2')...
  Best Params for this fold: {'featureSelection__score_func': <function f_regression at 0x000001BB62C9EC00>, 'svr__C': 1.25, 'svr__gamma': 'scale', 'svr__kernel': 'poly'}
  Calculating metrics on the outer test set...
    R2: 0.1180

Performing Outer Fold 2/5
Performing GridSearchCV (optimizing for 'r2')...
  Best Params for this fold: {'featureSelection__score_func': <function f_regression at 0x000001BB62C9EC00>, 'svr__C': 1.25, 'svr__gamma': 'auto', 'svr__kernel': 'poly'}
  Calculating metrics on the outer test set...
    R2: 0.2528

Performing Outer Fold 3/5
Performing GridSearchCV (optimizing for 'r2')...
  Best Params for this fold: {'featureSelection__score_func': <function f_regression at 0x000001BB62C9EC00>, 'svr__C': 1.25, 'svr__gamma': 'scale', 'svr__kernel': 'poly'}
  Calculating metrics on the outer test set...
    R2: 0.1766

Performing Outer Fold 4/5
Performing GridSearchCV (optimizing for 'r2')..

## Nested CV su Pipeline 2

In [22]:
start_time_gbr = time.time()

result_gbr = nested_cv(
    model = pipeline_gbr,
    param_grid = params_grid_gbr,
    X_train = X_train,
    y_train = y_train,
    outer_splits = 5,
    inner_splits = 3,
    scoring = ["r2"]
)

end_time_gbr = time.time()


Performing Outer Fold 1/5
Performing GridSearchCV (optimizing for 'r2')...
  Best Params for this fold: {'gbr__max_depth': 3, 'gbr__n_estimators': 75, 'pca__n_components': 3, 'pca__svd_solver': 'auto'}
  Calculating metrics on the outer test set...
    R2: 0.2679

Performing Outer Fold 2/5
Performing GridSearchCV (optimizing for 'r2')...
  Best Params for this fold: {'gbr__max_depth': 3, 'gbr__n_estimators': 75, 'pca__n_components': 5, 'pca__svd_solver': 'full'}
  Calculating metrics on the outer test set...
    R2: 0.4727

Performing Outer Fold 3/5
Performing GridSearchCV (optimizing for 'r2')...
  Best Params for this fold: {'gbr__max_depth': 3, 'gbr__n_estimators': 75, 'pca__n_components': 5, 'pca__svd_solver': 'full'}
  Calculating metrics on the outer test set...
    R2: 0.2344

Performing Outer Fold 4/5
Performing GridSearchCV (optimizing for 'r2')...
  Best Params for this fold: {'gbr__max_depth': 3, 'gbr__n_estimators': 75, 'pca__n_components': 5, 'pca__svd_solver': 'full'}
  

## Train final model from nested cross validation

In [23]:
from collections import Counter
from sklearn.base import clone
import numpy as np

def train_final_model_from_nested_cv(model,
                                     all_fold_best_params,
                                     X, y,
                                     score_per_fold: list[float] = None,
                                     strategy: str = 'most_frequent',
                                     X_test=None, y_test=None,
                                     scoring: list[str] = None,
                                     verbose=True):
    """
    Allena un modello finale usando i migliori iperparametri ottenuti da una nested CV,
    e calcola le metriche su un test set opzionale se fornito.

    Parameters:
        model: modello sklearn
        all_fold_best_params: lista dei parametri ottimali per ogni fold
        X, y: dati completi per l'addestramento
        score_per_fold: punteggi dei fold esterni, richiesto per 'best_fold'
        strategy: 'most_frequent' o 'best_fold'
        X_test, y_test: test set opzionale per calcolare le metriche finali
        scoring: lista di metriche da calcolare (default auto)
        verbose: se True, stampa info

    Returns:
        final_model: modello allenato su tutto il dataset
        best_params: iperparametri usati
        test_metrics
    """
    if not all_fold_best_params:
        raise ValueError("La lista di best_params è vuota.")

    if strategy == 'most_frequent':
        # Conta la combinazione più ricorrente tra i dizionari
        param_counts = Counter([frozenset(p.items()) for p in all_fold_best_params])
        most_common_params = dict(param_counts.most_common(1)[0][0])
        if verbose:
            print(f"\n[STRATEGIA: most_frequent] Parametri più frequenti sui fold:")
            print(most_common_params)
        best_params = most_common_params

    elif strategy == 'best_fold':
        if score_per_fold is None:
            raise ValueError("score_per_fold è richiesto per la strategia 'best_fold'.")
        if len(score_per_fold) != len(all_fold_best_params):
            raise ValueError("score_per_fold e all_fold_best_params devono avere la stessa lunghezza.")

        best_index = int(np.nanargmax(score_per_fold))
        best_params = all_fold_best_params[best_index]

        if verbose:
            print(f"\n[STRATEGIA: best_fold] Selezionato il fold #{best_index + 1} con punteggio migliore: {score_per_fold[best_index]:.4f}")
            print(f"Parametri selezionati: {best_params}")

    else:
        raise ValueError("Strategia non supportata: usa 'most_frequent' o 'best_fold'.")

    # Clona il modello e imposta i parametri selezionati
    final_model = clone(model).set_params(**best_params)

    # Allena su tutto il dataset
    final_model.fit(X, y)

    test_metrics = {}
    # Calcolo delle metriche su test set, se fornito
    if X_test is not None and y_test is not None:
        if verbose:
            print("\nCalcolo delle metriche sul test set finale...")

        y_pred = final_model.predict(X_test)
        try:
            y_proba = final_model.predict_proba(X_test)[:, 1]
        except:
            y_proba = None

        # Determina task
        est_type = getattr(final_model, "_estimator_type", None)
        is_clf = (est_type == "classifier")

        if scoring is None:
            scoring = ['accuracy', 'roc_auc'] if is_clf else ['r2', 'mae', 'rmse']

        for metric in scoring:
            if metric == 'accuracy':
                test_metrics['accuracy'] = accuracy_score(y_test, y_pred)
            elif metric == 'roc_auc':
                if y_proba is not None:
                    test_metrics['roc_auc'] = roc_auc_score(y_test, y_proba)
                else:
                    test_metrics['roc_auc'] = np.nan
            elif metric == 'r2':
                test_metrics['r2'] = r2_score(y_test, y_pred)
            elif metric == 'mae':
                test_metrics['mae'] = mean_absolute_error(y_test, y_pred)
            elif metric == 'rmse':
                test_metrics['rmse'] = mean_squared_error(y_test, y_pred, squared=False)
            else:
                test_metrics[metric] = np.nan

        if verbose:
            for m, v in test_metrics.items():
                print(f"  {m.upper()}: {v:.4f}")

    return final_model, best_params, test_metrics


## Final results for Pipeline 1

In [27]:
# Alleno il modello finale usando la strategia "best_fold"
final_model_svr, final_params_svr, test_metrics_svr = train_final_model_from_nested_cv(
    model=pipeline_svr,
    all_fold_best_params= result_svr["all_fold_best_params"],
    X=X_train,
    y=y_train,
    X_test=X_test,
    y_test=y_test,
    score_per_fold = result_svr["performance_summary"]["r2"]["all_scores"],
    strategy = 'most_frequent',
    scoring=['r2']
)


[STRATEGIA: most_frequent] Parametri più frequenti sui fold:
{'svr__kernel': 'poly', 'svr__C': 1.25, 'svr__gamma': 'scale', 'featureSelection__score_func': <function f_regression at 0x000001BB62C9EC00>}

Calcolo delle metriche sul test set finale...
  R2: 0.2862


## Final results for pipeline 2

In [26]:
# Alleno il modello finale usando la strategia "best_fold"
final_model_gbr, final_params_gbr, test_metrics_gbr = train_final_model_from_nested_cv(
    model=pipeline_gbr,
    all_fold_best_params= result_gbr["all_fold_best_params"],
    X=X_train,
    y=y_train,
    X_test=X_test,
    y_test=y_test,
    score_per_fold = result_gbr["performance_summary"]["r2"]["all_scores"],
    strategy = 'most_frequent',
    scoring=['r2']
)


[STRATEGIA: most_frequent] Parametri più frequenti sui fold:
{'gbr__n_estimators': 75, 'gbr__max_depth': 3, 'pca__n_components': 5, 'pca__svd_solver': 'full'}

Calcolo delle metriche sul test set finale...
  R2: 0.4481
